<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Facit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Städrobot - FACIT

> **⚠️ DETTA ÄR LÄRARVERSIONEN (FACIT) – innehåller alla svar och lösningar**

Den här notebooken innehåller:
- ✅ Alla svar på quiz-frågor
- ✅ Alla övningar genomförda med fungerande kod
- ✅ Kommentarer som förklarar varje lösning
- ✅ Alternativa lösningar där det är relevant

**Upphovspersoner:**
Originalversion: Mattias Tiger, mattias.tiger@liu.se

Förenklad version på svenska: baserad på originalverket ovan.
Facit: sammanställt för lärarstöd.

**Licens:** CC BY-NC-SA 4.0
https://creativecommons.org/licenses/by-nc-sa/4.0/


# Städrobot - Lab 1.0 - Rutnätsvärlden
Vi börjar med att sätta upp miljön: **Rutnätsvärlden**.

---

## SVAR: Uppgift 1.0.1 – Fråga 1
**Fråga:** Hur många celltyper finns det?

**✅ Svar: 5 stycken**
- `EMPTY = 0` – Tom cell (`.`)
- `WALL = 1` – Vägg (`#`)
- `DIRT = 2` – Smuts (`░`)
- `HOME = 3` – Hem (`H`)
- `UNKNOWN = 4` – Okänd cell (`?`) – används i Lab 1.3–1.4

*Eleverna hittar svaret i `class CellType` i kodcellen nedan.*


In [ ]:
import ipywidgets as widgets           # Verktyg för att skapa knappar och gränssnitt
from IPython.display import display, clear_output  # Verktyg för att visa och rensa utskrifter
import random                           # Slumptalsgenerator
import math                             # Matematikfunktioner

output = widgets.Output()


In [ ]:
# Typer av celler i rutnätet
class CellType:
  EMPTY   = 0  # Tom cell
  WALL    = 1  # Vägg
  DIRT    = 2  # Smuts
  HOME    = 3  # Robotens hem (startposition)
  UNKNOWN = 4  # Okänd cell (roboten har inte besökt den ännu)

def cell_to_string(cell):
  if cell == CellType.EMPTY:   return "."
  if cell == CellType.WALL:    return "#"
  if cell == CellType.DIRT:    return "░"
  if cell == CellType.HOME:    return "H"
  if cell == CellType.UNKNOWN: return "?"
  return f"Fel, ingen cell kallas {cell}"

def generate_world(grid_size, wall_density, dirt_density):
  area = (grid_size - 2) ** 2
  grid = [[CellType.EMPTY] * grid_size for _ in range(grid_size)]
  for row in range(grid_size):
    grid[row][0] = CellType.WALL
    grid[row][grid_size - 1] = CellType.WALL
  for col in range(grid_size):
    grid[0][col] = CellType.WALL
    grid[grid_size - 1][col] = CellType.WALL
  num_walls = math.ceil(area * wall_density)
  for _ in range(num_walls):
    row, col, tries = 1, 1, 1000
    while grid[row][col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)
      col = random.randint(1, grid_size - 2)
      tries -= 1
    if tries > 0:
      grid[row][col] = CellType.WALL
  num_dirt = math.ceil(area * dirt_density)
  for _ in range(num_dirt):
    row, col, tries = 1, 1, 1000
    while grid[row][col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)
      col = random.randint(1, grid_size - 2)
      tries -= 1
    if tries > 0:
      grid[row][col] = CellType.DIRT
  return grid

def generate_scenario(grid_size, wall_density, dirt_density, random_home=False):
  grid = generate_world(grid_size, wall_density, dirt_density)
  if random_home:
    x, y, tries = 1, 1, 1000
    while grid[y][x] != CellType.EMPTY and tries > 0:
      x = random.randint(1, grid_size - 2)
      y = random.randint(1, grid_size - 2)
      tries -= 1
  else:
    x, y = 1, 1
  grid[y][x] = CellType.HOME
  return grid, (x, y)

def print_grid(world, robot_location_x=None, robot_location_y=None, margin=""):
  for y, row in enumerate(world):
    line = margin
    for x, cell in enumerate(row):
      if robot_location_x == x and robot_location_y == y:
        line += "R "
      else:
        line += cell_to_string(cell) + " "
    print(line)

def render_grid(output_widget, world, robot_pos=None):
  output_widget.clear_output()
  with output_widget:
    if robot_pos:
      print_grid(world, robot_pos[0], robot_pos[1])
    else:
      print_grid(world)

def print_grid_indices(world, robot_x=None, robot_y=None, with_symbol=False):
  for y, row in enumerate(world):
    line = ""
    for x, cell in enumerate(row):
      if with_symbol:
        sym = "R" if (robot_x == x and robot_y == y) else cell_to_string(cell)
        line += f"({x},{y},{sym}) "
      else:
        line += f"({x},{y}) "
    print(line)


## SVAR: Uppgift 1.0.3 – Frågor 2, 3, 4

**Fråga 2:** Hur stor är rutnätsvärlden?
**✅ Svar: 10 × 10** – definieras av `grid_size = 10`

**Fråga 3:** Vad händer om du ändrar `wall_density`?
**✅ Svar:** Fler `#`-symboler syns (fler väggar). `0.0` = inga väggar, `1.0` = nästan allt är väggar.

**Fråga 4:** Vad händer om du ändrar `dirt_density`?
**✅ Svar:** Fler `░`-symboler syns (mer smuts). Fungerar på samma sätt som `wall_density`.


In [ ]:
# LÖSNING: Övning 1.0.3 – Generera och visualisera ett scenario
grid_size    = 10   # 10×10 rutnät
wall_density = 0.1  # 10% väggar
dirt_density = 0.3  # 30% smuts
random_home  = False

grid, robot_pos = generate_scenario(grid_size, wall_density, dirt_density, random_home)

output_grid = widgets.Output()
render_grid(output_grid, grid, robot_pos)
print("Rutnätsvärlden (utskrift av miljön):")
display(output_grid)

# Tips: Prova att ändra wall_density till 0.5 och se vad som händer!


## SVAR: Uppgift 1.0.4 – Fråga 5

**Fråga 5:** Hur ändras y nedåt? Är det som förväntat?
**✅ Svar:** `y` **ökar nedåt** (y=0 är överst, desto längre ned man går desto större y-värde).
Det är bakvänt jämfört med matematikens koordinatsystem (där y ökar uppåt),
men är standardkonventionen för skärmar – skärmar räknar pixlar uppifrån.

```
  0 1 2 3 ...  ← x ökar höger
0 # # # #
1 # H . .
2 # . ░ .
↑
y ökar nedåt
```


In [ ]:
# LÖSNING: Övning 1.0.4 – Visa koordinater
print("Platsen (x,y) för varje rutnätscell:")
print_grid_indices(grid)
print()
print("Platsen för varje rutnätscell tillsammans med dess celltyp (x,y,typ):")
print_grid_indices(grid, robot_pos[0], robot_pos[1], with_symbol=True)


---
# Städrobot - Lab 1.1 - Simulering av en probleminstans

## SVAR: Uppgift 1.1.1 – Fråga 1

**Fråga:** Vilka åtgärder kan städroboten utföra?
**✅ Svar: 6 åtgärder:**
1. `NOOP` (0) – Gör ingenting
2. `MOVE_LEFT` (1) – Rör sig vänster
3. `MOVE_DOWN` (2) – Rör sig nedåt
4. `MOVE_RIGHT` (3) – Rör sig höger
5. `MOVE_UP` (4) – Rör sig uppåt
6. `SUCK_DIRT` (5) – Dammsuga smuts

*Eleverna hittar svaret i `AGENT_ACTION_*` konstanterna i kodcellen nedan.*


In [ ]:
# Definiera alla agentåtgärder
AGENT_ACTION_NOOP        = 0  # Gör ingenting
AGENT_ACTION_MOVE_LEFT   = 1  # Rör sig vänster
AGENT_ACTION_MOVE_DOWN   = 2  # Rör sig nedåt
AGENT_ACTION_MOVE_RIGHT  = 3  # Rör sig höger
AGENT_ACTION_MOVE_UP     = 4  # Rör sig uppåt
AGENT_ACTION_SUCK_DIRT   = 5  # Dammsug smuts

AGENT_ACTIONS = [
  AGENT_ACTION_NOOP, AGENT_ACTION_MOVE_UP, AGENT_ACTION_MOVE_DOWN,
  AGENT_ACTION_MOVE_LEFT, AGENT_ACTION_MOVE_RIGHT, AGENT_ACTION_SUCK_DIRT
]
AGENT_MOVE_ACTIONS = AGENT_ACTIONS[1:-1]

def action_to_string(action):
  if action == AGENT_ACTION_NOOP:       return "NOOP"
  if action == AGENT_ACTION_MOVE_UP:    return "UPP"
  if action == AGENT_ACTION_MOVE_DOWN:  return "NER"
  if action == AGENT_ACTION_MOVE_LEFT:  return "VÄNSTER"
  if action == AGENT_ACTION_MOVE_RIGHT: return "HÖGER"
  if action == AGENT_ACTION_SUCK_DIRT:  return "DAMMSUG"
  return f"Fel, ingen åtgärd kallas {action}"

class Environment:
  def __init__(self, grid_size, wall_density, dirt_density):
    self.world = generate_world(grid_size, wall_density, dirt_density)
    self.agent_location_x = 1
    self.agent_location_y = 1
    self.world[self.agent_location_y][self.agent_location_x] = CellType.HOME
    self.messages = []

  def percept(self, agent):
    return {
      "home": self.agent_location_x == 1 and self.agent_location_y == 1,
      "dirt": self.world[self.agent_location_y][self.agent_location_x] == CellType.DIRT,
      "bump": agent.bump
    }

  def is_bump(self, x, y):
    return self.world[y][x] == CellType.WALL

  def execute(self, agent, action):
    agent.bump = False
    agent.state.last_action = action
    x = self.agent_location_x
    y = self.agent_location_y
    if action == AGENT_ACTION_SUCK_DIRT and self.world[y][x] == CellType.DIRT:
      agent.performance += 100.0
      self.messages.append("Poäng +100")
    else:
      agent.performance -= 1.0
      self.messages.append("Poäng -1")
    if action == AGENT_ACTION_MOVE_DOWN:
      agent.bump = self.is_bump(x, y + 1)
      if not agent.bump: self.agent_location_y += 1
    elif action == AGENT_ACTION_MOVE_UP:
      agent.bump = self.is_bump(x, y - 1)
      if not agent.bump: self.agent_location_y -= 1
    elif action == AGENT_ACTION_MOVE_LEFT:
      agent.bump = self.is_bump(x - 1, y)
      if not agent.bump: self.agent_location_x -= 1
    elif action == AGENT_ACTION_MOVE_RIGHT:
      agent.bump = self.is_bump(x + 1, y)
      if not agent.bump: self.agent_location_x += 1
    elif action == AGENT_ACTION_SUCK_DIRT:
      if self.world[y][x] == CellType.DIRT:
        self.world[y][x] = CellType.EMPTY

class AgentStateBase:
  def __init__(self):
    self.location_x  = 1
    self.location_y  = 1
    self.last_action = AGENT_ACTION_NOOP

class Agent:
  def __init__(self):
    self.state       = AgentStateBase()
    self.bump        = False
    self.performance = 0.0
    self.last_action = AGENT_ACTION_NOOP

class RemoteCleanerAgent(Agent):
  def execute(self, percept):
    return AGENT_ACTION_NOOP

class SimulationBase:
  def __init__(self, environment, agent, output=None, environment_output=None, render_world=True):
    self.environment        = environment
    self.agent              = agent
    self.step_count         = 0
    self.render_world       = render_world
    self.status_output      = output
    self.environment_output = environment_output
    self.agent.performance  = -1000
    self.agent.last_action  = AGENT_ACTION_NOOP
    self.environment.messages.append(f"åtgärd: {action_to_string(self.agent.last_action)}")
    self.environment.messages.append(f"position (miljö): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")

  def is_mission_complete(self):
    for y, row in enumerate(self.environment.world):
      for x, cell in enumerate(row):
        if cell == CellType.DIRT: return False
    if self.environment.agent_location_x != 1 or self.environment.agent_location_y != 1:
      return False
    return True

  def send_manual_command(self, manual_action):
    with output:
      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})"); return
      self.step_count += 1
      percept = self.environment.percept(self.agent)
      self.agent.last_action = manual_action
      self.environment.execute(self.agent, manual_action)
      self.agent.state.location_x = self.environment.agent_location_x
      self.agent.state.location_y = self.environment.agent_location_y
      self.render_status()
      if self.environment_output: self.render_environment()
      if self.is_mission_complete(): print(f"Uppdrag slutfört (poäng: {self.agent.performance})")

  def step(self, user_feedback=True):
    with output:
      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})"); return
      self.step_count += 1
      percept = self.environment.percept(self.agent)
      action  = self.agent.execute(percept)
      self.environment.execute(self.agent, action)
      if user_feedback: self.render()
      if self.is_mission_complete(): print(f"Uppdrag slutfört (poäng: {self.agent.performance})")

  def render_environment(self):
    if self.render_world:
      render_grid(self.environment_output, self.environment.world,
                  (self.environment.agent_location_x, self.environment.agent_location_y))
    else:
      self.environment_output.clear_output()
    with self.environment_output:
      for line in self.environment.messages: print(line)
      self.environment.messages.clear()

  def render_status(self):
    if self.status_output:
      self.status_output.clear_output()
      with self.status_output:
        print(f"Iteration: {self.step_count}")
        print(f"Poäng: {self.agent.performance}")

  def render(self):
    self.render_status()
    self.render_environment()


## SVAR: Uppgift 1.1.2 – Frågor 2 och 3

**Fråga 2:** Vad är startpoängen?
**✅ Svar: -1000**

Förklaring: Startpoängen är negativt för att markera att uppdraget inte är slutfört.
Om roboten städar all smuts (+100 per smuts) och rör sig effektivt (-1 per rörelse) kan poängen bli positiv.

**Fråga 3:** Vad händer med poängen?
**✅ Svar:**
- Varje rörelse: **-1 poäng**
- Lyckad dammsugning (smuts finns): **+100 poäng**
- Misslyckad dammsugning (ingen smuts): **-1 poäng**

```python
# Poängberäkning – exempelkalkyl:
# Startpoäng: -1000
# 5 smutsar städade: +500
# 30 rörelser: -30
# Slutpoäng: -1000 + 500 - 30 = -530
```

## SVAR: Uppgift 1.1.3 – Fråga 4

**Fråga 4:** Vilken poäng fick du?
**✅ Svar: Varierar!** Beror på världen och hur effektivt roboten styrs.
Typisk poäng: ca -600 till -800. Positiv poäng kräver många smutsar och få rörelser.


In [ ]:
# LÖSNING: Uppgift 1.1.2 – Starta simuleringen med manuell styrning
# (Kör denna cell och använd knapparna för att styra roboten)

upp_knapp     = widgets.Button(description="Upp")
ner_knapp     = widgets.Button(description="Ner")
vänster_knapp = widgets.Button(description="Vänster")
höger_knapp   = widgets.Button(description="Höger")
dammsug_knapp = widgets.Button(description="Dammsug")

environment_world = widgets.Output()
status = widgets.Output()

environment = Environment(10, 0, 0.1)
agent       = RemoteCleanerAgent()
simulation  = SimulationBase(environment, agent, output=status, environment_output=environment_world)
simulation.render()
output.clear_output()

def klick_upp(b):     simulation.send_manual_command(AGENT_ACTION_MOVE_UP)
def klick_ner(b):     simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN)
def klick_vänster(b): simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT)
def klick_höger(b):   simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT)
def klick_dammsug(b): simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT)

upp_knapp.on_click(klick_upp)
ner_knapp.on_click(klick_ner)
vänster_knapp.on_click(klick_vänster)
höger_knapp.on_click(klick_höger)
dammsug_knapp.on_click(klick_dammsug)

knappar = widgets.HBox([vänster_knapp, widgets.VBox([upp_knapp, ner_knapp]), höger_knapp, dammsug_knapp])
display(widgets.HBox([environment_world, status]))
display(knappar)


---
# Städrobot - Lab 1.2 - Lös problemet i känd miljö

## SVAR: Uppgift 1.2.1 – Fråga 1

**Fråga:** Hur många grannceller kan en cell ha maximalt?
**✅ Svar: 4 stycken** – upp, höger, ner, vänster (inte diagonalt)

Faktiskt antal grannar kan vara 0–4 beroende på väggar och rutnätsgränser.


In [ ]:
# LÖSNING: Uppgift 1.2.1 – Hitta grannar och initiera miljö
grid_size    = 10
wall_density = 0.3
dirt_density = 0.02
random.seed(8)  # Fast slumpfrö = samma värld varje gång
environment = Environment(grid_size, wall_density, dirt_density)

def get_neighbors(pos, world):
  grannar = []
  for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:  # upp, höger, ner, vänster
    x = pos[0] + dx
    y = pos[1] + dy
    if 0 <= x < len(world[0]) and 0 <= y < len(world) and world[y][x] != CellType.WALL:
      grannar.append((x, y))
  return grannar

# Test: Visa grannar för position (1,1)
new_location = (1, 1)
print_grid(environment.world, robot_location_x=new_location[0], robot_location_y=new_location[1])
print(f"Fria grannar till {new_location}: {get_neighbors(new_location, environment.world)}")
print(f"Antal grannar: {len(get_neighbors(new_location, environment.world))} (max möjligt: 4)")


## SVAR: Uppgift 1.2.4 – DFT (Djupet-först-traversering)

**Fråga 1:** Täcker DFT alla nåbara positioner?
**✅ Svar: Ja** – DFT-algoritmen besöker varje nåbar nod exakt en gång.

**Fråga 2:** Var slutar DFT?
**✅ Svar: Beror på världen** – t.ex. (3,1) i ett specifikt test.
DFT:s slutposition är icke-deterministisk – beror på grannarnas ordning.

**Fråga 3:** Kan roboten följa DFT-stigen?
**✅ Svar: Ja**, men DFT-stigen kan hoppa mellan icke-angränsande celler.
BFS används för att fylla i rörelserna mellan koordinaterna.

### Hur DFT fungerar:
```
DFT använder en STACK (LIFO: sist in, först ut)

Startar på S → lägger till grannar i stacken
→ tar den SISTA i stacken → utforskar djupet
→ fortsätter tills allt är besökt

Resultat: Besöker ALLA nåbara noder (men inte kortaste väg)
```


In [ ]:
# LÖSNING: DFT - Djupet-först-traversering
def dft(start_node, map):
  visited  = []
  frontier = [start_node]  # Stack (LIFO)
  path = []
  while frontier:
    last_node = frontier.pop(-1)  # Ta SISTA noden (stack-beteende)
    if last_node not in visited:
      path.append(last_node)
      visited.append(last_node)
      for neighbor in get_neighbors(last_node, map):
        frontier.append(neighbor)
  return path

# SearchState för BFS
class SearchState:
  def __init__(self, location_x, location_y, parent=None, last_action=None):
    self.location_x  = location_x
    self.location_y  = location_y
    self.last_action = last_action
    self.parent      = parent

  def __hash__(self):
    return hash((self.location_x, self.location_y))

  def __eq__(self, other):
    return self.location_x == other.location_x and self.location_y == other.location_y

def apply_action(action, search_state):
  new_state = SearchState(search_state.location_x, search_state.location_y)
  if action == AGENT_ACTION_MOVE_UP:    new_state.location_y -= 1
  if action == AGENT_ACTION_MOVE_DOWN:  new_state.location_y += 1
  if action == AGENT_ACTION_MOVE_LEFT:  new_state.location_x -= 1
  if action == AGENT_ACTION_MOVE_RIGHT: new_state.location_x += 1
  return new_state

def bfs(start_state, world, goal_condition):
  visited  = []
  frontier = [(start_state, [])]  # Kö (FIFO)
  while frontier:
    state, plan = frontier.pop(0)  # Ta FÖRSTA i kön (kö-beteende)
    if goal_condition(state, world):
      return plan
    if state not in visited:
      visited.append(state)
      for action in AGENT_MOVE_ACTIONS:
        new_state = apply_action(action, state)
        if 0 <= new_state.location_x < len(world[0]) and 0 <= new_state.location_y < len(world):
          if world[new_state.location_y][new_state.location_x] != CellType.WALL:
            new_state.parent = state
            new_state.last_action = action
            frontier.append((new_state, plan + [action]))
  return []

# LÖSNING: Uppgift 1.2.4 – Kör DFT och se sökvägen
grid_size    = 6
dirt_density = 0.08
environment  = Environment(grid_size, 0.1, dirt_density)

print_grid(environment.world, robot_location_x=1, robot_location_y=1)
path = dft((1, 1), environment.world)
print("DFT-stig:", path)
print("Stiglängd (antal noder):", len(path))
print("Antal steg:", len(path) - 1)


## SVAR: Uppgift 1.2.5 – Plan från DFT-stig

**Fråga 1:** Kan roboten följa stigen med plan?
**✅ Svar: Ja** – BFS fyller i de saknade rörelserna.

**Fråga 2:** Hur mycket längre är planen?
**✅ Svar:** Planen är **längre** än stigen.
DFT-stigen räknar noder; planen räknar enskilda åtgärder (HÖGER, UPP osv).
En DFT-stig med 15 noder kan kräva 20–30 åtgärder.


In [ ]:
# LÖSNING: Uppgift 1.2.5 – Skapa plan från DFT-stig
plan = []
for n in range(1, len(path)):
  p = bfs(SearchState(path[n-1][0], path[n-1][1]),
          environment.world,
          lambda state, world: state.location_x == path[n][0] and state.location_y == path[n][1])
  plan.extend(p)

print("Plan (heltal):", plan)
print("Plan (text):", [action_to_string(a) for a in plan])
print("Planlängd (antal åtgärder):", len(plan))
print("DFT-stiglängd (antal noder):", len(path))
print(f"Planen är {len(plan) - (len(path)-1)} steg längre än DFT-stigens antal hopp")


## SVAR: Uppgift 1.2.7 – BFS till närmaste smuts

**Fråga 1:** Var hamnar du om du följer planen?
**✅ Svar: På den närmaste smutspositionen** från robotens startposition.

**Fråga 2:** Vad är kortaste plan till nästa smuts?
**✅ Svar: Beror på världen** – t.ex. `[UPP, HÖGER, UPP]` i ett test.

**Fråga 3:** Tar planerna roboten till närmaste smuts?
**✅ Svar: Ja** – BFS garanterar kortaste vägen.


In [ ]:
# LÖSNING: Uppgift 1.2.7 – BFS till närmaste smuts
robot_location = (1, 1)  # Prova olika startpositioner!

print_grid(environment.world, robot_location_x=robot_location[0], robot_location_y=robot_location[1])

plan = bfs(SearchState(robot_location[0], robot_location[1]),
           environment.world,
           lambda state, world: world[state.location_y][state.location_x] == CellType.DIRT)

print(f"Plan till närmaste smuts: {[action_to_string(a) for a in plan]}")
print(f"Antal steg: {len(plan)}")

# Beräkna slutposition
x, y = robot_location
for action in plan:
  if action == AGENT_ACTION_MOVE_UP:    y -= 1
  elif action == AGENT_ACTION_MOVE_DOWN:  y += 1
  elif action == AGENT_ACTION_MOVE_LEFT:  x -= 1
  elif action == AGENT_ACTION_MOVE_RIGHT: x += 1
print(f"Slutposition efter att följa planen: ({x}, {y})")
print(f"Är det smuts där? {environment.world[y][x] == CellType.DIRT}")


## SVAR: Uppgift 1.2.9 – Slumpmässig agent vs. BFS-agent

**Fråga 1:** Är det lätt/svårt för slumpmässig agent?
**✅ Svar: Mycket svårt** – Den väljer slumpmässiga åtgärder och hittar inte smutsen effektivt.
Typiskt: Tar hundratals/tusentals steg utan att lösa problemet.

**Fråga 2:** Vad gör BFS-agenten bättre?
**✅ Svar:** BFS-agenten:
1. Vet var all smuts finns (känd miljö)
2. Beräknar **kortaste vägen** direkt till varje smuts
3. Städar systematiskt all smuts
4. Återvänder hem via kortaste väg
Resultat: Löser problemet på optimalt antal åtgärder!


In [ ]:
# LÖSNING: Uppgift 1.2.9 – AgentState och BFS-planeringsagent

class AgentState(AgentStateBase):
  def __init__(self, world_max_width, world_max_height):
    AgentStateBase.__init__(self)
    self.world = [[CellType.UNKNOWN] * world_max_width for _ in range(world_max_height)]
    self.world[1][1] = CellType.HOME

  def update_location(self, percept):
    if self.last_action == AGENT_ACTION_MOVE_UP and not percept["bump"]:
      self.location_y -= 1
    elif self.last_action == AGENT_ACTION_MOVE_DOWN and not percept["bump"]:
      self.location_y += 1
    elif self.last_action == AGENT_ACTION_MOVE_LEFT and not percept["bump"]:
      self.location_x -= 1
    elif self.last_action == AGENT_ACTION_MOVE_RIGHT and not percept["bump"]:
      self.location_x += 1
    if percept.get("bump", False):
      if self.last_action == AGENT_ACTION_MOVE_UP:
        self.world[self.location_y-1][self.location_x] = CellType.WALL
      elif self.last_action == AGENT_ACTION_MOVE_DOWN:
        self.world[self.location_y+1][self.location_x] = CellType.WALL
      elif self.last_action == AGENT_ACTION_MOVE_LEFT:
        self.world[self.location_y][self.location_x-1] = CellType.WALL
      elif self.last_action == AGENT_ACTION_MOVE_RIGHT:
        self.world[self.location_y][self.location_x+1] = CellType.WALL
    if percept.get("dirt", False):
      self.world[self.location_y][self.location_x] = CellType.DIRT
    elif self.world[self.location_y][self.location_x] not in [CellType.HOME, CellType.WALL]:
      self.world[self.location_y][self.location_x] = CellType.EMPTY

class RandomCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)
    self.state = AgentState(10, 10)

  def execute(self, percept):
    self.state.update_location(percept)
    return random.choice(AGENT_ACTIONS)

class BfsPlanningCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)
    self.state = AgentState(10, 10)
    self.plan  = []

  def execute(self, percept):
    self.state.update_location(percept)
    if percept.get("dirt", False):
      self.plan = []
      return AGENT_ACTION_SUCK_DIRT
    if not self.plan:
      # Sök efter närmaste smuts
      self.plan = bfs(
        SearchState(self.state.location_x, self.state.location_y),
        self.state.world,
        lambda state, world: world[state.location_y][state.location_x] == CellType.DIRT
      )
    if not self.plan:
      # Ingen smuts kvar – gå hem
      self.plan = bfs(
        SearchState(self.state.location_x, self.state.location_y),
        self.state.world,
        lambda state, world: state.location_x == 1 and state.location_y == 1
      )
    if self.plan:
      return self.plan.pop(0)
    return AGENT_ACTION_NOOP

print("✅ BfsPlanningCleanerAgent definierad")
print("Kör simulering nedan för att testa agenten")


In [ ]:
# LÖSNING: Uppgift 1.2.9 – Starta simulering med BFS-agent

class Simulation(SimulationBase):
  def __init__(self, environment, agent, output=None, environment_output=None,
               agent_world_output=None, render_world=True):
    SimulationBase.__init__(self, environment, agent, output, environment_output, render_world)
    self.agent_world_output = agent_world_output

  def render_environment(self):
    if self.agent_world_output:
      render_grid(self.agent_world_output, self.agent.state.world,
                  (self.agent.state.location_x, self.agent.state.location_y))
    SimulationBase.render_environment(self)

steg_knapp  = widgets.Button(description="Steg")
steg_10     = widgets.Button(description="Steg10")
steg_100    = widgets.Button(description="Steg100")
återställ   = widgets.Button(description="Återställ")

agenter = [('Slumpmässig', 'random'), ('BFS-planering', 'bfs')]
agent_dropdown = widgets.Dropdown(options=agenter, value='bfs', description='Robot')

environment_world = widgets.Output()
agent_world       = widgets.Output()
status            = widgets.Output()

def create_simulation(agent_type):
  env = Environment(10, 0.1, 0.1)
  if agent_type == 'random':
    agt = RandomCleanerAgent()
  else:
    agt = BfsPlanningCleanerAgent()
  return Simulation(env, agt, output=status,
                    environment_output=environment_world,
                    agent_world_output=agent_world)

simulation = create_simulation('bfs')
simulation.render()
output.clear_output()

def on_steg(b):
  simulation.step()
def on_steg_10(b):
  for _ in range(10): simulation.step(user_feedback=False)
  simulation.render()
def on_steg_100(b):
  for _ in range(100): simulation.step(user_feedback=False)
  simulation.render()
def on_återställ(b):
  global simulation
  simulation = create_simulation(agent_dropdown.value)
  simulation.render()

steg_knapp.on_click(on_steg)
steg_10.on_click(on_steg_10)
steg_100.on_click(on_steg_100)
återställ.on_click(on_återställ)

display(widgets.HBox([
  widgets.VBox([widgets.HTML('<b>Miljö (verklig)</b>'), environment_world]),
  widgets.VBox([widgets.HTML('<b>Agentens karta</b>'), agent_world]),
  widgets.VBox([widgets.HTML('<b>Status</b>'), status])
]))
display(widgets.HBox([steg_knapp, steg_10, steg_100, återställ, agent_dropdown]))


---
# Städrobot - Lab 1.3 - Okänd miljö

## SVAR: Uppgift 1.3.1 – Frågor 1, 2, 3

**Fråga 1:** Är det svårare med okänd miljö?
**✅ Svar: Ja** – Roboten vet inte var smutsen eller väggarna är. Den måste utforska.

**Fråga 2:** Kan vi använda gamla BFS-lösningen?
**✅ Svar: Nej** – BFS söker efter DIRT i agentens karta. I okänd miljö är alla celler märkta
som UNKNOWN (?) så BFS hittar ingen smuts att gå till.

**Fråga 3:** Vad är en bra strategi?
**✅ Svar: Systematisk utforskning** – Besök varje UNKNOWN cell i ordning.
Precis som DFT besöker alla nåbara noder, men nu i realtid med okänd karta.


In [ ]:
# LÖSNING: Lab 1.3 – Fjärrstyrd agent i okänd miljö
class CleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)
    self.state = AgentState(10, 10)

  def execute(self, percept):
    self.state.update_location(percept)
    return AGENT_ACTION_NOOP

upp_knapp     = widgets.Button(description="Upp")
ner_knapp     = widgets.Button(description="Ner")
vänster_knapp = widgets.Button(description="Vänster")
höger_knapp   = widgets.Button(description="Höger")
dammsug_knapp = widgets.Button(description="Dammsug")

environment_world = widgets.Output()
agent_world       = widgets.Output()
status            = widgets.Output()

environment = Environment(10, 0.1, 0.1)
agent       = CleanerAgent()
simulation  = Simulation(environment, agent, output=status,
                         environment_output=environment_world,
                         agent_world_output=agent_world,
                         render_world=False)  # Dölj verklig värld!
simulation.render()
output.clear_output()

def klick_upp(b):     simulation.send_manual_command(AGENT_ACTION_MOVE_UP)
def klick_ner(b):     simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN)
def klick_vänster(b): simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT)
def klick_höger(b):   simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT)
def klick_dammsug(b): simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT)

upp_knapp.on_click(klick_upp)
ner_knapp.on_click(klick_ner)
vänster_knapp.on_click(klick_vänster)
höger_knapp.on_click(klick_höger)
dammsug_knapp.on_click(klick_dammsug)

display(widgets.HBox([
  widgets.VBox([widgets.HTML('<b>Agentens karta (okänd)</b>'), agent_world]),
  widgets.VBox([widgets.HTML('<b>Status</b>'), status])
]))
display(widgets.HBox([vänster_knapp, widgets.VBox([upp_knapp, ner_knapp]), höger_knapp, dammsug_knapp]))


---
# Städrobot - Lab 1.4 - Lös problemet i okänd miljö

## SVAR: Uppgift 1.4.1 – Frågor 1 och 2

**Fråga 1:** Är det svårt för slumpmässig agent i okänd miljö?
**✅ Svar: Ja, mycket svårt** – Samma problem som i känd miljö, men nu vet agenten inte ens var väggarna är.

**Fråga 2:** Vad gör BFS-planeringsagenten bättre?
**✅ Svar:** BFS-agenten söker systematiskt efter **UNKNOWN celler** istället för smuts.
Strategi:
1. Om det finns smuts här → städa
2. Om det finns smuts i känd karta → BFS dit
3. Annars → BFS till närmaste UNKNOWN cell (utforska)
4. Om allt utforskat → gå hem

Agenten lär sig gradvis om världen och löser problemet systematiskt!


In [ ]:
# LÖSNING: BFS-planeringsagent för okänd miljö
class BfsPlanningUnknownCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)
    self.state = AgentState(10, 10)
    self.plan  = []

  def execute(self, percept):
    self.state.update_location(percept)

    # 1. Om det finns smuts här – städa genast!
    if percept.get("dirt", False):
      self.plan = []
      return AGENT_ACTION_SUCK_DIRT

    # 2. Om vi inte har en plan – skapa en ny
    if not self.plan:
      # Försök hitta känd smuts
      self.plan = bfs(
        SearchState(self.state.location_x, self.state.location_y),
        self.state.world,
        lambda state, world: world[state.location_y][state.location_x] == CellType.DIRT
      )

    if not self.plan:
      # Ingen känd smuts – utforska närmaste okända cell
      self.plan = bfs(
        SearchState(self.state.location_x, self.state.location_y),
        self.state.world,
        lambda state, world: world[state.location_y][state.location_x] == CellType.UNKNOWN
      )

    if not self.plan:
      # Allt är utforskat – gå hem!
      self.plan = bfs(
        SearchState(self.state.location_x, self.state.location_y),
        self.state.world,
        lambda state, world: state.location_x == 1 and state.location_y == 1
      )

    if self.plan:
      return self.plan.pop(0)
    return AGENT_ACTION_NOOP

print("✅ BfsPlanningUnknownCleanerAgent definierad")


In [ ]:
# LÖSNING: Starta simulering med BFS-agent i okänd miljö
steg_knapp  = widgets.Button(description="Steg")
steg_10     = widgets.Button(description="Steg10")
steg_100    = widgets.Button(description="Steg100")
återställ   = widgets.Button(description="Återställ")

agenter_14 = [('Slumpmässig', 'random'), ('BFS-okänd', 'bfs_unknown')]
agent_dropdown_14 = widgets.Dropdown(options=agenter_14, value='bfs_unknown', description='Robot')

env_world_14   = widgets.Output()
agent_world_14 = widgets.Output()
status_14      = widgets.Output()

def create_sim_14(agent_type):
  env = Environment(10, 0.1, 0.1)
  agt = BfsPlanningUnknownCleanerAgent() if agent_type == 'bfs_unknown' else RandomCleanerAgent()
  return Simulation(env, agt, output=status_14,
                    environment_output=env_world_14,
                    agent_world_output=agent_world_14,
                    render_world=False)

sim_14 = create_sim_14('bfs_unknown')
sim_14.render()
output.clear_output()

def on_steg_14(b): sim_14.step()
def on_steg10_14(b):
  for _ in range(10): sim_14.step(user_feedback=False)
  sim_14.render()
def on_steg100_14(b):
  for _ in range(100): sim_14.step(user_feedback=False)
  sim_14.render()
def on_reset_14(b):
  global sim_14
  sim_14 = create_sim_14(agent_dropdown_14.value)
  sim_14.render()

steg_knapp.on_click(on_steg_14)
steg_10.on_click(on_steg10_14)
steg_100.on_click(on_steg100_14)
återställ.on_click(on_reset_14)

display(widgets.HBox([
  widgets.VBox([widgets.HTML('<b>Agentens karta</b>'), agent_world_14]),
  widgets.VBox([widgets.HTML('<b>Status</b>'), status_14])
]))
display(widgets.HBox([steg_knapp, steg_10, steg_100, återställ, agent_dropdown_14]))


---
# Sammanfattning – Alla svar

| Del | Fråga | Svar |
|-----|-------|------|
| **1.0** | Antal celltyper? | **5** (tom, vägg, smuts, hem, okänd) |
| **1.0** | Storlek på världen? | **10×10** |
| **1.0** | wall_density effekt? | **Fler/färre väggar** |
| **1.0** | dirt_density effekt? | **Mer/mindre smuts** |
| **1.0** | y-axeln? | **Ökar nedåt** (skärmkonvention) |
| **1.1** | Antal åtgärder? | **6** (NOOP, UPP, NER, VÄN, HÖG, DAMMSUG) |
| **1.1** | Startpoäng? | **-1000** |
| **1.1** | Poängändring? | **-1 rörelse, +100 dammsugning** |
| **1.2** | Max grannar? | **4** |
| **1.2** | DFT besöker allt? | **Ja** |
| **1.2** | Slumpmässig agent bra? | **Nej, mycket ineffektiv** |
| **1.2** | BFS-agent fördel? | **Planerar kortaste väg** |
| **1.3** | Okänd miljö svårare? | **Ja** |
| **1.3** | Bra strategi? | **Systematisk utforskning (UNKNOWN-sökning)** |
| **1.4** | BFS i okänd miljö? | **Söker UNKNOWN-celler systematiskt** |

---
*Facit – Lab 1 Städrobot | CC BY-NC-SA 4.0*
